# Manual Testing: Complete 3-Agent Pipeline with CredibilityFilter

This notebook provides hands-on testing of the expanded pipeline using real API keys:

## Pipeline Flow
1. **QueryOrchestrator** - Parses user query → structured search query + intent
2. **TavilyRetriever** - Executes Tavily two-step process (search → extract)
3. **CredibilityFilter** - Scores and filters results by source credibility
4. **Analysis Tools** - Interactive credibility analysis and filtering

## Credibility Scoring Components
- **Domain Reputation (50%)**: Authority and trustworthiness of source domain
- **Recency Score (30%)**: Content freshness with exponential decay
- **Extractability (20%)**: Quality of data extraction from Tavily

## Requirements
- Real OPENAI_API_KEY and TAVILY_API_KEY in .env file
- Backend dependencies installed

## Setup and Imports

In [ ]:
import os
import sys
import asyncio
import json
from datetime import datetime, timezone, timedelta
from pprint import pprint
import pandas as pd
from urllib.parse import urlparse

# Add backend to path
sys.path.append('..')

# Load environment variables
from dotenv import load_dotenv
load_dotenv('../.env')

print("✅ Environment loaded")
print(f"OPENAI_API_KEY: {'✅ Set' if os.getenv('OPENAI_API_KEY') else '❌ Missing'}")
print(f"TAVILY_API_KEY: {'✅ Set' if os.getenv('TAVILY_API_KEY') else '❌ Missing'}")

In [ ]:
# Import our agents and state management
from app.agents.query_orchestrator_agent import QueryOrchestratorAgent
from app.agents.tavily_retriever_agent import TavilyRetrieverAgent
from app.agents.credibility_filter_agent import CredibilityFilterAgent
from app.agents.state import create_initial_state, get_state_summary
from app.config import settings
from app.extractors.domain_config import get_domain_quality_score

print("✅ Agents imported successfully")
print(f"Environment: {settings.ENVIRONMENT}")
print(f"OpenAI Model: {settings.OPENAI_MODEL}")

## Initialize Agents

In [ ]:
# Create agent instances
query_agent = QueryOrchestratorAgent()
tavily_agent = TavilyRetrieverAgent()
credibility_agent = CredibilityFilterAgent()

print("✅ Agents initialized:")
print(f"- {query_agent.name} (OpenAI: {query_agent.llm.model_name})")
print(f"- {tavily_agent.name} (Client: {type(tavily_agent.tavily_client).__name__})")
print(f"- {credibility_agent.name} (Threshold: {credibility_agent.credibility_threshold})")

## Test Configuration

In [ ]:
# Test Tavily configuration
config_test = await tavily_agent.test_configuration()
print("🔧 Tavily Configuration:")
pprint(config_test)

# Show credibility filter configuration
print("\n🎯 CredibilityFilter Configuration:")
print(f"  Primary threshold: {credibility_agent.credibility_threshold}")
print(f"  Fallback thresholds: {credibility_agent.fallback_thresholds}")
print(f"  Min results fallback: {credibility_agent.min_results_fallback}")
print(f"  Intent weights available: {list(credibility_agent.intent_weights.keys())}")

## Complete Pipeline Test Functions

In [ ]:
async def test_complete_pipeline(raw_query: str, run_id: str = None):
    """
    Complete 3-agent pipeline test: QueryOrchestrator → TavilyRetriever → CredibilityFilter
    """
    if not run_id:
        run_id = f"manual_test_{datetime.now().strftime('%H%M%S')}"
    
    print(f"\n🚀 Testing Complete Pipeline: '{raw_query}'")
    print("=" * 70)
    
    # Create initial state
    state = create_initial_state(raw_query=raw_query, run_id=run_id)
    print(f"📋 Initial state created (run_id: {run_id})")
    
    try:
        # Step 1: QueryOrchestrator
        print("\n1️⃣ QueryOrchestrator Processing...")
        start_time = datetime.now()
        state = await query_agent.process(state)
        query_time = (datetime.now() - start_time).total_seconds()
        
        search_query = state.get("search_query")
        if search_query:
            print(f"✅ Query parsed successfully ({query_time:.2f}s)")
            print(f"   Intent: {search_query.intent}")
            print(f"   Category: {search_query.category}")
            print(f"   Normalized: '{search_query.normalized_query}'")
            if search_query.budget_max:
                print(f"   Budget: ${search_query.budget_max}")
            if search_query.constraints:
                print(f"   Constraints: {search_query.constraints}")
        else:
            print("❌ Query parsing failed")
            return state
        
        # Step 2: TavilyRetriever
        print("\n2️⃣ TavilyRetriever Processing...")
        start_time = datetime.now()
        state = await tavily_agent.process(state)
        tavily_time = (datetime.now() - start_time).total_seconds()
        
        search_results = state.get("raw_search_results", [])
        extracted_content = state.get("extracted_content", [])
        coverage_score = state.get("coverage_score", 0.0)
        
        print(f"✅ Tavily processing completed ({tavily_time:.2f}s)")
        print(f"   Search results: {len(search_results)}")
        print(f"   Extracted content: {len(extracted_content)}")
        print(f"   Coverage score: {coverage_score:.2f}")
        
        # Step 3: CredibilityFilter
        print("\n3️⃣ CredibilityFilter Processing...")
        start_time = datetime.now()
        state = await credibility_agent.process(state)
        credibility_time = (datetime.now() - start_time).total_seconds()
        
        filtered_results = state.get("credibility_filtered_results", [])
        
        print(f"✅ Credibility filtering completed ({credibility_time:.2f}s)")
        print(f"   Filtered results: {len(filtered_results)} (from {len(search_results)})")
        
        if filtered_results:
            scores = [r.get("credibility_score", 0) for r in filtered_results]
            avg_score = sum(scores) / len(scores)
            print(f"   Average credibility: {avg_score:.3f}")
            print(f"   Score range: {min(scores):.3f} - {max(scores):.3f}")
        
        # Show agent execution summary
        summary = get_state_summary(state)
        total_time = query_time + tavily_time + credibility_time
        print(f"\n📊 Pipeline Summary:")
        print(f"   Total time: {total_time:.2f}s")
        print(f"   Agents completed: {summary['progress']['agents_completed']}")
        print(f"   Total cost: ${summary['progress']['total_cost_usd']:.4f}")
        
        return state
        
    except Exception as e:
        print(f"❌ Pipeline error: {e}")
        import traceback
        traceback.print_exc()
        return state

def analyze_credibility_scores(state: dict, detailed: bool = True):
    """
    Analyze credibility scores in detail
    """
    filtered_results = state.get("credibility_filtered_results", [])
    
    if not filtered_results:
        print("No filtered results to analyze")
        return
    
    print(f"\n🔍 Credibility Analysis ({len(filtered_results)} results):")
    print("=" * 60)
    
    for i, result in enumerate(filtered_results, 1):
        url = result.get("url", "No URL")
        domain = urlparse(url).netloc.replace("www.", "")
        title = result.get("title", "No title")
        score = result.get("credibility_score", 0)
        breakdown = result.get("credibility_breakdown", {})
        
        print(f"\n{i}. {title[:60]}...")
        print(f"   Domain: {domain}")
        print(f"   Overall Score: {score:.3f}")
        
        if detailed and breakdown:
            print(f"   📊 Breakdown:")
            print(f"      Domain Score: {breakdown.get('domain_score', 0):.3f}")
            print(f"      Recency Score: {breakdown.get('recency_score', 0):.3f}")
            print(f"      Extractability: {breakdown.get('extractability_score', 0):.3f}")
            print(f"      Intent: {breakdown.get('intent', 'unknown')}")
        
        # Show warnings if any
        if "credibility_warning" in result:
            print(f"   ⚠️ Warning: {result['credibility_warning']}")

def compare_before_after_filtering(state: dict):
    """
    Compare results before and after credibility filtering
    """
    raw_results = state.get("raw_search_results", [])
    filtered_results = state.get("credibility_filtered_results", [])
    
    print(f"\n📊 Before/After Filtering Comparison:")
    print("=" * 50)
    print(f"Raw results: {len(raw_results)}")
    print(f"Filtered results: {len(filtered_results)}")
    print(f"Filtering ratio: {len(filtered_results)/max(len(raw_results), 1)*100:.1f}%")
    
    if filtered_results:
        # Show which domains made it through
        filtered_domains = [urlparse(r.get("url", "")).netloc.replace("www.", "") for r in filtered_results]
        domain_counts = {}
        for domain in filtered_domains:
            domain_counts[domain] = domain_counts.get(domain, 0) + 1
        
        print(f"\n🏆 Domains that passed filtering:")
        for domain, count in sorted(domain_counts.items(), key=lambda x: x[1], reverse=True):
            domain_score = get_domain_quality_score(domain)
            print(f"   {domain}: {count} results (domain score: {domain_score:.3f})")

def show_filtering_thresholds_impact(state: dict):
    """
    Show how different filtering thresholds would impact results
    """
    filtered_results = state.get("credibility_filtered_results", [])
    
    if not filtered_results:
        return
    
    scores = [r.get("credibility_score", 0) for r in filtered_results]
    thresholds = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
    
    print(f"\n🎯 Impact of Different Filtering Thresholds:")
    print("=" * 50)
    
    for threshold in thresholds:
        passing = sum(1 for score in scores if score >= threshold)
        percentage = passing / len(scores) * 100
        indicator = "🟢" if threshold <= 0.4 else "🟡" if threshold <= 0.6 else "🔴"
        current = " (CURRENT)" if threshold == 0.4 else ""
        print(f"   {indicator} Threshold {threshold:.1f}: {passing}/{len(scores)} results ({percentage:.1f}%){current}")

## Interactive Testing

Now you can test different queries manually and analyze the credibility filtering results!

### Test 1: Product Search - Wireless Earbuds

In [ ]:
# Test product search with budget constraint
result_state = await test_complete_pipeline("best wireless earbuds under $100")

In [ ]:
# Analyze the credibility scores in detail
analyze_credibility_scores(result_state, detailed=True)

In [ ]:
# Compare before/after filtering
compare_before_after_filtering(result_state)

In [ ]:
# Show impact of different thresholds
show_filtering_thresholds_impact(result_state)

### Test 2: Review Search - Product Reviews

In [ ]:
# Test review search (different intent weighting)
review_state = await test_complete_pipeline("MacBook Pro M4 review 2025")

In [ ]:
# Analyze review search credibility
analyze_credibility_scores(review_state, detailed=True)
compare_before_after_filtering(review_state)

### Test 3: Comparison Search - Product Comparison

In [ ]:
# Test comparison search (emphasizes extractability)
comparison_state = await test_complete_pipeline("iPhone 15 vs Samsung Galaxy S24 camera quality")

In [ ]:
# Analyze comparison search credibility
analyze_credibility_scores(comparison_state, detailed=True)
show_filtering_thresholds_impact(comparison_state)

### Custom Query Testing

Use this cell to test your own queries and analyze the credibility filtering:

In [ ]:
# Your custom query here
custom_query = "gaming monitor 4K 144Hz under $500"
custom_result = await test_complete_pipeline(custom_query)

In [ ]:
# Analyze your custom results
analyze_credibility_scores(custom_result, detailed=True)
compare_before_after_filtering(custom_result)
show_filtering_thresholds_impact(custom_result)

## Advanced Analysis: Intent-Aware Scoring

In [ ]:
def compare_intent_scoring():
    """
    Show how different intents affect credibility weighting
    """
    print("🎯 Intent-Aware Scoring Weights:")
    print("=" * 40)
    
    for intent, weights in credibility_agent.intent_weights.items():
        print(f"\n{intent.upper()}:")
        print(f"   Domain: {weights['domain']*100:.0f}%")
        print(f"   Recency: {weights['recency']*100:.0f}%")
        print(f"   Extractability: {weights['extractability']*100:.0f}%")
        
        # Explain the rationale
        if intent == "product_search":
            print(f"   💡 Emphasizes domain authority (e-commerce trust)")
        elif intent == "review_search":
            print(f"   💡 Balanced approach (fresh reviews from trusted sources)")
        elif intent == "comparison":
            print(f"   💡 Emphasizes extractability (structured comparison data)")

compare_intent_scoring()

## Performance Analysis

In [ ]:
# Performance test with multiple queries
performance_queries = [
    "laptop for programming",
    "smartphone camera review", 
    "tablet vs laptop comparison",
    "wireless headphones noise canceling"
]

performance_results = []

print("⚡ Performance Testing:")
print("=" * 30)

for i, query in enumerate(performance_queries, 1):
    print(f"\n{i}. Testing: '{query}'")
    start_time = datetime.now()
    
    result = await test_complete_pipeline(query, f"perf_test_{i}")
    
    total_time = (datetime.now() - start_time).total_seconds()
    summary = get_state_summary(result)
    
    filtered_count = len(result.get("credibility_filtered_results", []))
    raw_count = len(result.get("raw_search_results", []))
    
    performance_results.append({
        "query": query,
        "time_seconds": total_time,
        "cost_usd": summary['progress']['total_cost_usd'],
        "raw_results": raw_count,
        "filtered_results": filtered_count,
        "filter_ratio": filtered_count / max(raw_count, 1)
    })

print("\n📊 Performance Summary:")
print("=" * 50)
df = pd.DataFrame(performance_results)
print(df.to_string(index=False, float_format=lambda x: f'{x:.2f}'))

print(f"\n🎯 Performance Metrics:")
print(f"   Average time: {df['time_seconds'].mean():.1f}s")
print(f"   Average cost: ${df['cost_usd'].mean():.4f}")
print(f"   Average filter ratio: {df['filter_ratio'].mean():.1%}")

## Deep Dive: Domain Quality Analysis

In [ ]:
# Analyze domain quality scores for common domains
common_domains = [
    "amazon.com", "bestbuy.com", "walmart.com", "target.com",
    "wirecutter.nytimes.com", "cnet.com", "techradar.com",
    "pcmag.com", "tomsguide.com", "digitaltrends.com",
    "reddit.com", "quora.com", "unknown-site.com"
]

print("🏆 Domain Quality Scores:")
print("=" * 40)

domain_scores = []
for domain in common_domains:
    score = get_domain_quality_score(domain)
    domain_scores.append({"domain": domain, "score": score})
    
    # Visual indicator
    if score >= 0.9:
        indicator = "🟢 Premium"
    elif score >= 0.8:
        indicator = "🟡 High"
    elif score >= 0.7:
        indicator = "🟠 Medium"
    else:
        indicator = "🔴 Low"
        
    print(f"   {indicator:12} {domain:25} {score:.3f}")

# Show distribution
df_domains = pd.DataFrame(domain_scores)
print(f"\n📊 Domain Score Distribution:")
print(f"   Mean: {df_domains['score'].mean():.3f}")
print(f"   Std: {df_domains['score'].std():.3f}")
print(f"   Range: {df_domains['score'].min():.3f} - {df_domains['score'].max():.3f}")

## Experimental: Custom Threshold Testing

In [ ]:
def test_custom_threshold(state: dict, custom_threshold: float):
    """
    Test filtering with a custom threshold
    """
    filtered_results = state.get("credibility_filtered_results", [])
    
    if not filtered_results:
        print("No results to test with custom threshold")
        return
    
    # Apply custom threshold
    custom_filtered = [r for r in filtered_results if r.get("credibility_score", 0) >= custom_threshold]
    
    print(f"\n🎯 Custom Threshold Testing (≥{custom_threshold}):")
    print(f"   Original results: {len(filtered_results)}")
    print(f"   Custom filtered: {len(custom_filtered)}")
    print(f"   Retention rate: {len(custom_filtered)/len(filtered_results)*100:.1f}%")
    
    if custom_filtered:
        avg_score = sum(r.get("credibility_score", 0) for r in custom_filtered) / len(custom_filtered)
        print(f"   Average score: {avg_score:.3f}")
        
        print(f"\n🏆 Results passing custom threshold:")
        for i, result in enumerate(custom_filtered[:3], 1):
            domain = urlparse(result.get("url", "")).netloc.replace("www.", "")
            score = result.get("credibility_score", 0)
            title = result.get("title", "No title")[:50]
            print(f"   {i}. {domain} ({score:.3f}) - {title}...")

# Test with different thresholds
if 'custom_result' in locals():
    test_custom_threshold(custom_result, 0.6)
    test_custom_threshold(custom_result, 0.8)
else:
    print("Run a custom query first to test custom thresholds")

## Final Summary & Next Steps

🎉 **Congratulations!** You've successfully tested the complete 3-agent pipeline:

### ✅ **What We've Tested:**
1. **QueryOrchestrator** - Intent recognition and query parsing
2. **TavilyRetriever** - Real web search and content extraction
3. **CredibilityFilter** - Multi-factor credibility scoring and filtering

### 🎯 **Key Findings:**
- **Intent-aware scoring** adapts weights based on search type
- **Domain reputation** strongly influences filtering decisions
- **Recency scoring** favors fresh content with exponential decay
- **Extractability** rewards structured, information-rich content
- **Progressive fallback** ensures results even when quality is low

### ➡️ **Next Agent: SpecExtractorAgent**
The pipeline is ready for the next component that will:
- Extract structured product data from credibility-filtered results
- Use Tavily extraction + LLM enhancement for missing fields
- Validate against product schemas
- Calculate field coverage scores

The foundation is solid and ready for structured data extraction! 🚀